# pavia4 — DARTS with std vs pcaN fronts, vs LRao strong recipe
Question: does the rotation-only front (mild-PCA truncation, NO per-component
rescale — the one front safe on every setting) add value for DARTS on pavia4?
Compared against the LRao STRONG recipe (no cutoff, robust median/IQR input
norm, detach sigma, FULL-batch steps, [128] relu, wd 0, no clip, fixed
budget, no early stopping) on identical planted targets.
Noise-before convention throughout. Saves fp16 snaps at every eval,
best+final states, final score maps (plain + CFAR), full curves.
Edit CONFIG, run top to bottom.

In [ ]:
!git clone -b camera-ready --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/pavia-u.mat'), 'missing data'

In [ ]:
# ======================= CONFIG — edit me =======================
CONFIG = {
  'target_cls': None,      # None = bitumen protocol sig | 5 = metal sheets
  'theta': 0.15,           # bitumen .15 | metal .075
  'seeds': [42, 43],
  # ---- DARTS arms: front x rho ----
  'fronts': ['std', 'pcaN99.999', 'pcaN99.9999'],
  'rhos': [0.03, 0.5],
  'epochs': 10000,
  'darts': {'d_lat': 16, 'K': 7, 'enc_hidden': [64, 32],
            'score_hidden': [128], 'activation': 'relu',
            'lr': 3e-4, 'weight_decay': 1e-5, 'grad_clip': 1.0,
            'batch_size': 512},
  'eval_every': 200,
  # ---- LRao strong recipe ----
  'lrao': {'epochs': 5000, 'eval_every': 100, 'hidden': [128],
           'lr': 5e-4, 'delta_theta': 0.01},
}
# ================================================================
import json
json.dump(CONFIG, open('dp_config.json', 'w'), indent=1)
print(json.dumps(CONFIG, indent=1))

In [ ]:
%%writefile run_darts_fronts.py
"""pavia4 DARTS front comparison — std vs pcaN (rotation-only reduction).
The pcaN front pre-projects EVERYTHING (train, test, planted, neighbor
windows, signature) onto the top-m PCs (orthonormal -> isotropic noise
stays isotropic), then uses a single global scalar inside the frozen
whitening. Config from dp_config.json. Outputs: results_dp.json +
ckpt_dp/<key>/."""
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
from tqdm import tqdm

from repro import scenes
from repro.scenes import pavia_protocol as PP
from repro.protocols.spatial import load_cfg
from repro.core.data import Whitening, plant_targets, extract_neighborhoods
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.seeding import seed_all
from repro.models.darts.model import DARTS, _NeighborDenoiser

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
CFG = json.load(open('dp_config.json'))
THETA = float(CFG['theta'])
TARGET_CLS = CFG['target_cls']
TAG = 'p4' if TARGET_CLS is None else f'p4cls{TARGET_CLS}'
EPOCHS = int(CFG['epochs'])
EVAL_EVERY = int(CFG['eval_every'])
OUT_JSON = 'results_dp.json'
CKPT_ROOT = 'ckpt_dp'

SP_CFG = load_cfg()
os.makedirs(CKPT_ROOT, exist_ok=True)
_SC = {}


def get_scene():
    if not _SC:
        sc = scenes.build('pavia4', SP_CFG)
        if TARGET_CLS is not None:
            sc['sig_use'] = PP.foreign_signature(
                sc['data'], sc['gt'], sc['te'],
                cls=int(TARGET_CLS)).astype(np.float32)
        else:
            sc['sig_use'] = np.asarray(sc['sig'], np.float32)
        _SC.update(sc)
    return _SC


def windows_of(flat, shape, k):
    img = torch.tensor(np.asarray(flat, np.float32).reshape(*shape, -1))
    _, nbr = extract_neighborhoods(img, k)
    return nbr.numpy()


def build_space(front, sc, planted):
    """Project (or not) everything; return dict with reduced arrays + W."""
    tr, te, s = sc['tr'], sc['te'], sc['sig_use']
    if front == 'std':
        X = np.asarray(tr, np.float64)
        W = Whitening(X.mean(0).astype(np.float32),
                      np.diag(1.0 / X.std(0)).astype(np.float32))
        return dict(tr=np.asarray(tr, np.float32),
                    te=np.asarray(te, np.float32),
                    planted=np.asarray(planted, np.float32),
                    s=np.asarray(s, np.float32), W=W,
                    info=f'{tr.shape[1]} bands (std)')
    frac = float(front[4:]) / 100.0
    X = np.asarray(tr, np.float64)
    mu = X.mean(0)
    lam, V = np.linalg.eigh(np.cov(X, rowvar=False))
    lam, V = lam[::-1], V[:, ::-1]
    m = int(np.searchsorted(np.cumsum(lam) / lam.sum(), frac) + 1)
    P = V[:, :m].astype(np.float32)
    mu32 = mu.astype(np.float32)
    tr_r = (np.asarray(tr, np.float32) - mu32) @ P
    c = float(np.sqrt(np.asarray(tr_r, np.float64).var(0).mean()))
    W = Whitening(tr_r.mean(0).astype(np.float32),
                  (np.eye(m) / c).astype(np.float32))
    return dict(tr=tr_r,
                te=(np.asarray(te, np.float32) - mu32) @ P,
                planted=(np.asarray(planted, np.float32) - mu32) @ P,
                s=np.asarray(s, np.float32) @ P, W=W,
                info=f'{m}/{tr.shape[1]} PCs (noscale {frac*100:.4f}%)')


def run_one(front, rho, seed):
    key = f'{TAG}_darts_{front}_r{rho}_s{seed}'
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    if key in res:
        print('skip (done):', key); return
    t0 = time.time()
    sc = get_scene()
    planted0, labels, _ = plant_targets(
        sc['te'], sc['sig_use'], THETA, float(SP_CFG['target_fraction']),
        model='additive', seed=seed, spatial_shape=sc['te_shape'],
        edge_guard=int(SP_CFG['edge_guard']))
    y = np.asarray(labels)
    sp = build_space(front, sc, planted0)
    print(f'[{key}] {sp["info"]}', flush=True)
    d = sp['tr'].shape[1]
    k = int(SP_CFG['k'])
    tr_nbr = windows_of(sp['tr'], sc['tr_shape'], k)
    te_nbr = windows_of(sp['te'], sc['te_shape'], k)
    c = CFG['darts']
    sigma = float(np.sqrt(rho * np.asarray(sp['tr'], np.float64).var(0).mean()))
    seed_all(seed)
    net = _NeighborDenoiser(d, int(c['d_lat']), int(c['K']),
                            list(c['enc_hidden']), list(c['score_hidden']),
                            float(np.sqrt(SP_CFG['darts']['dsm_sigma_rho'])),
                            c['activation'], sp['W']).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=float(c['lr']),
                            weight_decay=float(c['weight_decay']))
    X = torch.tensor(sp['tr'], device=DEVICE)
    N = torch.tensor(tr_nbr, device=DEVICE)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
    P_, B = len(X), int(c['batch_size'])
    rundir = os.path.join(CKPT_ROOT, key)
    os.makedirs(rundir, exist_ok=True)
    s = sp['s']
    cwin = int(SP_CFG['darts_cfar_window'] or k)
    cguard = int(SP_CFG['darts_cfar_guard'])

    def score_all(pix, nbr):
        out = []
        with torch.no_grad():
            for i in range(0, len(pix), 1024):
                p = torch.tensor(np.asarray(pix[i:i+1024], np.float32),
                                 device=DEVICE)
                nb = torch.tensor(np.asarray(nbr[i:i+1024], np.float32),
                                  device=DEVICE)
                out.append(net(p, nb).cpu().numpy())
        return np.concatenate(out, 0)

    def evaluate():
        z_tr = score_all(sp['tr'], tr_nbr)
        z_te = score_all(sp['planted'], te_nbr)
        zb = z_tr.mean(0)
        C = np.cov(z_tr, rowvar=False)
        T = -((z_te - zb) @ s) / np.sqrt(float(s @ C @ s))
        Tc = DARTS.local_moment_normalize(
            T, sc['te_shape'], cwin, guard=cguard,
            cfar_lam=float(SP_CFG['cfar_lam']))
        return T, Tc

    curve, best = [], {'auc': -1.0}
    bar = tqdm(range(1, EPOCHS + 1), desc=key, ncols=130, mininterval=5.0,
               file=sys.stdout, ascii=True)
    for ep in bar:
        net.train()
        perm = torch.randperm(P_, generator=gen, device=DEVICE)
        for i in range(0, P_, B):
            sel = perm[i:i + B]
            eps = torch.randn((len(sel), d), generator=gen,
                              device=DEVICE) * sigma
            psi = net(X[sel] + eps, N[sel])
            loss = ((psi + eps / sigma ** 2) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),
                                           float(c['grad_clip']))
            opt.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            net.eval()
            T, Tc = evaluate()
            auc = float(auc_safe(y, T))
            row = {'epoch': ep, 'auc': round(auc, 4),
                   'pd05': round(float(dr_at_fpr(y, T,
                                                 fpr_list=(0.05,))['0.05']), 4),
                   'auc_cfar': round(float(auc_safe(y, Tc)), 4),
                   'pd05_cfar': round(float(dr_at_fpr(y, Tc,
                                                      fpr_list=(0.05,))['0.05']), 4)}
            curve.append(row)
            if auc > best['auc']:
                best = dict(row)
            torch.save({'w16': {k_: v.half().cpu() for k_, v in
                                net.state_dict().items()}, 'epoch': ep},
                       os.path.join(rundir, f'snap_{ep:06d}.pt'))
            bar.set_postfix_str(f'loss={float(loss.detach()):.3g} '
                                f'auc={auc:.3f} cfar={row["auc_cfar"]:.3f} '
                                f'best={best["auc"]:.3f}@{best["epoch"]}')
    bar.close()
    net.eval()
    T, Tc = evaluate()
    np.savez_compressed(os.path.join(rundir, 'scores_final.npz'),
                        T=T, T_cfar=Tc, labels=y)
    torch.save({'net_final': {k_: v.cpu() for k_, v in
                              net.state_dict().items()}},
               os.path.join(rundir, 'model.pt'))
    out = {'det': 'darts', 'front': front, 'tag': TAG, 'rho': rho,
           'seed': seed, 'theta': THETA, 'epochs_done': EPOCHS,
           'dim': d, 'info': sp['info'], 'best': best, 'final': curve[-1],
           'curve': curve, 'sec': round(time.time() - t0)}
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    res[key] = out
    json.dump(res, open(OUT_JSON, 'w'), indent=1)
    print(f'[{key}] best_auc={best["auc"]:.3f}@{best["epoch"]} '
          f'final={curve[-1]} ({out["sec"]}s)', flush=True)


if __name__ == '__main__':
    done = set(json.load(open(OUT_JSON)).keys()) if os.path.exists(OUT_JSON) else set()
    tasks = [(f, float(r), sd) for f in CFG['fronts'] for r in CFG['rhos']
             for sd in [int(x) for x in CFG['seeds']]
             if f'{TAG}_darts_{f}_r{float(r)}_s{sd}' not in done]
    print(f'{len(tasks)} DARTS tasks, {EPOCHS} ep, device={DEVICE}',
          flush=True)
    for f, r, sd in tasks:
        run_one(f, r, sd)
    print('ALL DONE', flush=True)

In [ ]:
# ---- run the DARTS arms (fronts x rhos x seeds) ----
!python run_darts_fronts.py

In [ ]:
# ---- LRao STRONG recipe on the same cell (no cutoff, robust IQR,
# detach sigma, FULL batch, no ES) ----
import json, os, time
import numpy as np, torch
from tqdm import tqdm
from run_darts_fronts import get_scene, SP_CFG, THETA, TAG
from repro.core.data import plant_targets
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.models import (ScoreNet, lfi_loss_mode2,
                               compute_lfi_detector_scores_mode2)
from repro.core.normalization import robust_whitening_iqr
from repro.models.darts.model import DARTS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CFG = json.load(open('dp_config.json'))
L = CFG['lrao']
res = json.load(open('results_dp.json')) if os.path.exists('results_dp.json') else {}
sc = get_scene()
tr = np.asarray(sc['tr'], np.float32)
cwin = int(SP_CFG['darts_cfar_window'] or SP_CFG['k'])
for seed in [int(s) for s in CFG['seeds']]:
    key = f'{TAG}_lrao_strong_s{seed}'
    if key in res:
        print('skip (done):', key); continue
    t0 = time.time()
    planted, labels, _ = plant_targets(
        sc['te'], sc['sig_use'], THETA, float(SP_CFG['target_fraction']),
        model='additive', seed=seed, spatial_shape=sc['te_shape'],
        edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32); y = np.asarray(labels)
    Wl = robust_whitening_iqr(tr)
    torch.manual_seed(seed)
    model = ScoreNet(tr.shape[1], list(L['hidden']), 'relu',
                     whitening=Wl).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=float(L['lr']),
                           weight_decay=0.0)
    Xl = torch.tensor(tr, device=DEVICE)
    curve, best = [], {'pd05': -1.0}
    bar = tqdm(range(1, int(L['epochs']) + 1), desc=key, ncols=120,
               mininterval=5.0, ascii=True)
    for ep in bar:
        model.train()
        try:
            loss = lfi_loss_mode2(model, Xl, float(L['delta_theta']),
                                  detach_sigma=True)
            ok = torch.isfinite(loss)
        except Exception:
            ok = False
        if ok:
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % int(L['eval_every']) == 0 or ep == int(L['epochs']):
            model.eval()
            T = np.asarray(compute_lfi_detector_scores_mode2(
                model, tr, planted, sc['sig_use'], float(L['delta_theta'])))
            Tc = DARTS.local_moment_normalize(
                T, sc['te_shape'], cwin,
                guard=int(SP_CFG['darts_cfar_guard']),
                cfar_lam=float(SP_CFG['cfar_lam']))
            row = {'epoch': ep, 'auc': round(float(auc_safe(y, T)), 4),
                   'pd05': round(float(dr_at_fpr(y, T,
                                                 fpr_list=(0.05,))['0.05']), 4),
                   'auc_cfar': round(float(auc_safe(y, Tc)), 4),
                   'pd05_cfar': round(float(dr_at_fpr(y, Tc,
                                                      fpr_list=(0.05,))['0.05']), 4)}
            curve.append(row)
            if row['pd05'] > best['pd05']:
                best = dict(row)
            bar.set_postfix_str(f"auc={row['auc']:.3f} pd05={row['pd05']:.3f} "
                                f"best={best['pd05']:.3f}@{best['epoch']}")
    bar.close()
    res[key] = {'det': 'lrao_strong', 'tag': TAG, 'seed': seed,
                'theta': THETA, 'best': best, 'final': curve[-1],
                'curve': curve, 'sec': round(time.time() - t0)}
    json.dump(res, open('results_dp.json', 'w'), indent=1)
    print(f'[{key}] best={best} final={curve[-1]}', flush=True)

In [ ]:
# ---- Summary: DARTS fronts vs LRao strong ----
import json
import numpy as np
r = json.load(open('results_dp.json'))
groups = {}
for k, v in r.items():
    if v['det'] == 'darts':
        lab = f"DARTS {v['front']} rho={v['rho']}"
    else:
        lab = 'LRao strong'
    groups.setdefault(lab, []).append(v)
print(f"{'model':<28} {'best_auc':>8} {'final_auc':>9} {'final_cfar':>10} {'final_pd05':>10}")
for lab, vs in sorted(groups.items()):
    b = np.mean([v['best']['auc'] for v in vs])
    fa = np.mean([v['final']['auc'] for v in vs])
    fc = np.mean([v['final']['auc_cfar'] for v in vs])
    fp = np.mean([v['final']['pd05'] for v in vs])
    print(f'{lab:<28} {b:>8.3f} {fa:>9.3f} {fc:>10.3f} {fp:>10.3f}')

In [ ]:
# ---- Archive EVERYTHING ----
import shutil, os
!zip -q -r dp_results.zip results_dp.json dp_config.json ckpt_dp
print(os.path.getsize('dp_results.zip')/1e6, 'MB')
from google.colab import files
shutil.copy('dp_results.zip', 'dp_results_dl.zip')
files.download('dp_results_dl.zip')